# Real vs Fake Faces

We are going to use `Fake-Vs-Real-Faces` dataset. The Dataset contains two types of image which are: `Fake faces` image collected from the website [thispersondoesnotexist.com](thispersondoesnotexist.com). , `Real faces` images are re collected through the API of the website [Unsplash](https://unsplash.com/) and then the faces are cropped out of using OpenCV library.

Here some additional information about the dataset:
- Total number of images: 1288
- Number of "Fake" faces: 700
- Number of "Real" faces: 589

### 1. Import Libraries 

In [6]:
import torch 
import torch.nn as nn
from torch.nn import Linear, Dropout, ReLU
from torchvision import transforms
from torch.utils.data import DataLoader, Subset, random_split, Dataset
from torchvision.datasets import ImageFolder # Stream data from images stored in folders
import torch.optim as optim
import torch.nn.functional as F
# from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryF1Score
from torchinfo import summary

from sklearn.model_selection import StratifiedKFold, train_test_split
import os # allow s to access files
import numpy as np
from PIL import Image # Allows us to Load Images
from collections import Counter # Utility function to give us the counts of unique items in an iterable
import matplotlib.pyplot as plt
import shutil

## 2. Reconstruct the Folder structure of the Data

In [7]:
torch.manual_seed(42)

Before loading, let's reconstruct the data folder structure to give us comfort when using the `ImageFolder` class

In [8]:
def copy_image(des_dir, split_name, cls, file_list, cls_path):

    """
    des_dir: data_split
    split_name: can be "train" or "test"
    cls and cls_path: is class which can be "fake" or "real"
    file_list: list of images files

    save a copy of images from folder 'cls_path/img' into folder called 'des_dir/split_name/cls/img' 
    """
    
    for img in file_list:
        src = os.path.join(cls_path, img)
        dest = os.path.join(des_dir, split_name, cls, img)
        shutil.copy2(src, dest)

In [9]:

def split_and_save_to_disk(src_dir, des_dir):

    """
    reconstruct the folder structure 
    from:
        data/
            fake/
            real

    to:
        data_split/
            train/
                fake/
                real/
            test/
                fake/
                real/
    
    """
    
    # # splite ratios
    train_ratio = 0.8
    test_ratio = 0.2

    if os.path.exists(des_dir):
        print(f"{src_dir} already exists")
    else:
        print("reconstruct the data folder structure")

        # create folder 
        for split in ['train', 'test']:
            for cls in os.listdir(src_dir):
                os.makedirs(os.path.join(des_dir, split, cls), exist_ok=True)  # make it similar to this (des_dir/train/fake) and  (des_dir/train/real)


        # Process each class folder 
        for cls in os.listdir(src_dir):
            cls_path = os.path.join(src_dir, cls)
            if not os.path.isdir(cls_path):
                continue


            images = [f for f in os.listdir(cls_path) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

            # split train and (test and val)
            train_data, test_data = train_test_split(images, test_size=test_ratio, random_state=42)


            # splitting the data 
            print(f"splitting class '{cls}':\n train:{len(train_data)}\n test:{len(test_data)}")

            # des_dir, split_name, cls, file_list, cls_path
            copy_image(des_dir, "train", cls, train_data, cls_path)
            copy_image(des_dir, "test", cls, test_data, cls_path)
            
        print("Copying proces is finished")
        

## 3. Load and Prepocess the data

First Let's define the transforms pipeline for `data augmentatin` and `preprocessing` steps

In [10]:
split_and_save_to_disk("data", "data_split")

data already exists


In [11]:
mean=[0.485, 0.456, 0.406]
std=[0.229, 0.224, 0.225]

# for data preprocessing and data augmentation for trianing data
train_transforms = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])


test_valid_transforms = transforms.Compose([
    transforms.Resize((299, 299)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),

])

Let's load our images using `ImageFolder`. This class inherits from `DatasetFolder`  class. So, we can override the classes to create custom datasets as well. Because we need to custome dataset class becasue `ImageFolder` apply a single tansform on the whole dataset while we need to apply custome `transform` on each subset in the dataset.

In [12]:
path_to_train = './split_data/train'
path_to_test = './split_data/test'

train_data = ImageFolder(root=path_to_train, transform=None)
test_data = ImageFolder(root=path_to_test, transform=test_valid_transforms)

class CustomSubset(Dataset):
    def __init__(self, subset, transform=None):
        self.subset = subset
        self.transform = transform
    
    def __getitem__(self, idx):
        
        actual_idx = self.subset.indices[idx]
        
        image_path, label = self.subset.dataset.samples[actual_idx]
        image = self.subset.dataset.loader(image_path) 
        
        
        if self.transform:
            image = self.transform(image)
        else:
            image = transforms.ToTensor()(image)
        
        return image, label
    
    def __len__(self):
        return len(self.subset)
        
# Create the DataLoders
BATCH_SIZE = 16
# train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE, shuffle=True)

## 3. Build Helper functions

Before starting building the model we will build a helper function to save our models

In [13]:
def save_model(epochs, model, optimizer, loss_fun, model_name):

    """
    Function to save the trained model to disk
    """
    torch.save({
        'epoch': epochs,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss_fun,
    }, f'outputs/{model_name}.pth')

The next function is for saving the loss and accuracy plots after the training completes

In [14]:
def save_plots(train_acc, valid_acc, test_acc, train_loss, valid_loss, test_loss):

    """
    Function to save the loss and accurcy plots to disk
    """

    # accuracy plots
    plt.figure(figsize=(10, 7))
    plt.plot(train_acc, color='green', linestyle='-', label='train accuracy')
    plt.plot(valid_acc, color='red', linestyle='-', label='validation accuracy')
    plt.plot(test_acc, color='cyan', linestyle='-', label='test accuracy')
    plt.xlabel('EPOCHs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.savefig('outputs/accuracy.png')

    # loss plots
    plt.figure(figsize=(10, 7))
    plt.plot(train_loss, color='orange', linestyle='-', label='train loss')
    plt.plot(valid_loss, color='blue', linestyle='-', label='valid loss')
    plt.plot(test_loss, color='magenta', linestyle='-', label='test loss')
    plt.xlabel('EPOCHs')
    plt.ylabel('Loss')
    plt.legend()
    plt.savefig('outputs/loss.png')

## 4. Prepparing the Data Loaders and Stratified KFold

Now we will build our `data loader` and `stratified KFold` spliter for that we need for training.

In [15]:
N_SPLITs = 5
SEED = 42

In [16]:
def train(model, training_data, X, y, optimizer, loss_fn, Epochs = 10, N_SPLITs = 4):

    """
    Train the given model
    """


    
    skf = StratifiedKFold(n_splits=N_SPLITs, shuffle=True, random_state=SEED)
    stratified_scores = []

    # folds results 
    all_fold_results = []

    # Lists to collect metrics
    train_acc_list, valid_acc_list = [], []
    train_loss_list, valid_loss_list = [], []

    for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):



        # Initialize fold-specific metrics
        fold_train_loss = []
        fold_val_loss = []
        fold_train_acc = []
        fold_val_acc = []
        
        # Create the Data Loaders
        train_subset = Subset(training_data, train_idx)
        val_subset = Subset(training_data, valid_idx)


        # Apply transforms
        train_data_fold = CustomSubset(train_subset, transform=train_transforms)
        val_data_fold = CustomSubset(val_subset, transform=test_valid_transforms)
    
        # Dataloaders
        train_loader = DataLoader(train_data_fold, batch_size=BATCH_SIZE, shuffle=True)
        valid_loader = DataLoader(val_data_fold, batch_size=BATCH_SIZE, shuffle=False)

        ### Training ###
        model.train() 
        print("Training")
        for epoch in range(Epochs):

            train_loss, correct, total = 0.0, 0, 0
            
            for images, labels in train_loader:
                predictions = model(images)
                labels = labels.unsqueeze(1).float()
                loss = loss_fn(predictions, labels)
                loss.backward() # backpagation

                train_loss += loss.item()
                
                optimizer.step() # update the weight
                optimizer.zero_grad() # set the gradient to zero
                
                preds = (torch.sigmoid(predictions) > 0.5).float()
                correct += (preds == labels).sum().item()
                total += labels.size(0)


            # calculate the accuracy and loss for the current epoch
            train_accuracy = correct / total
            epoch_train_loss = train_loss / len(train_loader)


            ### Validation ###
            model.eval()
            print("Evaluting")
            val_loss, correct, total = 0.0, 0, 0
            with torch.no_grad():
                for images, labels in valid_loader:
                    predictions = model(images)
                    labels = labels.unsqueeze(1).float()
                    val_loss += loss_fn(predictions, labels).item()
                    
                    preds = (torch.sigmoid(predictions) > 0.5).float()
                    correct += (preds == labels).sum().item()
                    total += labels.size(0)

                    
            val_accuracy = correct / total
            epoch_val_loss = val_loss / len(valid_loader)
     

        
            fold_train_loss.append(train_loss)
            fold_val_loss.append(val_loss)
            fold_train_acc.append(train_accuracy)
            fold_val_acc.append(val_accuracy)
            
            print(f"Epoch {epoch+1}/{Epochs} - Train Loss: {epoch_train_loss:.4f}, Train Acc: {train_accuracy:.4f}, Val Loss: {epoch_val_loss:.4f}, Val Acc: {val_accuracy:.4f}")

            
        # Store metrics for this fold
        all_fold_results.append({
            'fold': fold + 1,
            'train_loss': fold_train_loss,
            'val_loss': fold_val_loss,
            'train_acc': fold_train_acc,
            'val_acc': fold_val_acc
        })


    avg_val_acc = sum(r['val_acc'][-1] for r in all_fold_results) / N_SPLITs
    avg_val_loss = sum(r['val_loss'][-1] for r in all_fold_results) / N_SPLITs
    print(f"\nAverage Validation Accuracy across folds: {avg_val_acc:.4f}")
    print(f"Average Validation Loss across folds: {avg_val_loss:.4f}")

    return all_fold_results
        
              
    

## 4, 5. Build and Train the Models

> ### 4.1 InceptionV3

In [17]:
# Load the AlexNet model and weights with pre-trian ImageNet weights
from torchvision.models import inception_v3
from torchvision.models import Inception_V3_Weights

# initialize the pretrained alexnet
inceptionv3 = inception_v3(weights=Inception_V3_Weights.DEFAULT)

inceptionv3.eval()

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

> ### 4.2 Freeze early layers 

In [18]:
inceptionv3.aux_logits = False

# for parameter in model.parameters():
#     parameter.requires_grad = False

for parameter in inceptionv3.parameters():
    parameter.requires_grad = False


> ### 4.3 Modfiy the inceptionv3 (output layer)

In [19]:
inceptionv3.fc = nn.Sequential(
    Linear(in_features=2048, out_features=1, bias=True),
)

> ### 4.4 Get model summary

In [20]:
summary(inceptionv3,
        input_size=(16, 3, 299, 299),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=18,
        row_settings=["var_names"]
)

Layer (type (var_name))                  Input Shape        Output Shape       Param #            Trainable
Inception3 (Inception3)                  [16, 3, 299, 299]  [16, 1]            3,326,696          Partial
├─BasicConv2d (Conv2d_1a_3x3)            [16, 3, 299, 299]  [16, 32, 149, 149] --                 False
│    └─Conv2d (conv)                     [16, 3, 299, 299]  [16, 32, 149, 149] (864)              False
│    └─BatchNorm2d (bn)                  [16, 32, 149, 149] [16, 32, 149, 149] (64)               False
├─BasicConv2d (Conv2d_2a_3x3)            [16, 32, 149, 149] [16, 32, 147, 147] --                 False
│    └─Conv2d (conv)                     [16, 32, 149, 149] [16, 32, 147, 147] (9,216)            False
│    └─BatchNorm2d (bn)                  [16, 32, 147, 147] [16, 32, 147, 147] (64)               False
├─BasicConv2d (Conv2d_2b_3x3)            [16, 32, 147, 147] [16, 64, 147, 147] --                 False
│    └─Conv2d (conv)                     [16, 32, 147, 147

> ### 4.4 Train the Model

In [21]:
# Specifiy learing rate and number of Epochs
inceptionv3_LR = 0.001
inceptionv3_Epochs = 2
inceptionv3_n_splits = 4
# Define loss and optimizer
inceptionv3_loss_fun = nn.BCEWithLogitsLoss()
inceptionv3_optimizer = torch.optim.Adam(inceptionv3.parameters(), lr=inceptionv3_LR)


train_X = [train_data.samples[img][0] for img in range(len(train_data.samples))]
train_y = [train_data.samples[label][1] for label in range(len(train_data.samples))]

skf = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)
BATCH_SIZE = 16

In [22]:
result = train(inceptionv3, train_data, train_X, train_y, inceptionv3_optimizer, inceptionv3_loss_fun, inceptionv3_Epochs, N_SPLITs = inceptionv3_n_splits)

Training
Evaluting
Epoch 1/2 - Train Loss: 0.5801, Train Acc: 0.7232, Val Loss: 0.5499, Val Acc: 0.6783
Evaluting
Epoch 2/2 - Train Loss: 0.3856, Train Acc: 0.8939, Val Loss: 0.4018, Val Acc: 0.8760
Training
Evaluting
Epoch 1/2 - Train Loss: 0.3743, Train Acc: 0.8396, Val Loss: 0.3830, Val Acc: 0.8062
Evaluting
Epoch 2/2 - Train Loss: 0.2633, Train Acc: 0.9314, Val Loss: 0.2642, Val Acc: 0.9186
Training
Evaluting
Epoch 1/2 - Train Loss: 0.3011, Train Acc: 0.9017, Val Loss: 0.2496, Val Acc: 0.9496
Evaluting
Epoch 2/2 - Train Loss: 0.2126, Train Acc: 0.9366, Val Loss: 0.2100, Val Acc: 0.9612
Training
Evaluting
Epoch 1/2 - Train Loss: 0.2508, Train Acc: 0.9173, Val Loss: 0.2130, Val Acc: 0.9494
Evaluting
Epoch 2/2 - Train Loss: 0.1576, Train Acc: 0.9703, Val Loss: 0.2311, Val Acc: 0.9183

Average Validation Accuracy across folds: 0.9185
Average Validation Loss across folds: 4.7052


> ### 5.1 InceptionV3 partial

In [23]:
# Load the AlexNet model and weights with pre-trian ImageNet weights
from torchvision.models import inception_v3
from torchvision.models import Inception_V3_Weights

# initialize the pretrained alexnet
inceptionv3_full = inception_v3(weights=Inception_V3_Weights.DEFAULT)

inceptionv3_full.eval()

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

> ### 5.2 Freeze early layers 

In [24]:
inceptionv3_full.aux_logits = False

for parameter in inceptionv3_full.parameters():
    parameter.requires_grad = True

> ### 5.3 Get model summary

In [25]:
summary(inceptionv3_full,
        input_size=(16, 3, 299, 299),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=18,
        row_settings=["var_names"]
)

Layer (type (var_name))                  Input Shape        Output Shape       Param #            Trainable
Inception3 (Inception3)                  [16, 3, 299, 299]  [16, 1000]         3,326,696          True
├─BasicConv2d (Conv2d_1a_3x3)            [16, 3, 299, 299]  [16, 32, 149, 149] --                 True
│    └─Conv2d (conv)                     [16, 3, 299, 299]  [16, 32, 149, 149] 864                True
│    └─BatchNorm2d (bn)                  [16, 32, 149, 149] [16, 32, 149, 149] 64                 True
├─BasicConv2d (Conv2d_2a_3x3)            [16, 32, 149, 149] [16, 32, 147, 147] --                 True
│    └─Conv2d (conv)                     [16, 32, 149, 149] [16, 32, 147, 147] 9,216              True
│    └─BatchNorm2d (bn)                  [16, 32, 147, 147] [16, 32, 147, 147] 64                 True
├─BasicConv2d (Conv2d_2b_3x3)            [16, 32, 147, 147] [16, 64, 147, 147] --                 True
│    └─Conv2d (conv)                     [16, 32, 147, 147] [16, 64,

> ### 5.4 Modfiy the InceptionV3 (output layer)

In [26]:
inceptionv3_full.fc = nn.Sequential(
    Linear(in_features=2048, out_features=1, bias=True),
)

> ### 5.5 Train the model

In [27]:
# Specifiy learing rate and number of Epochs
inceptionv3_full_LR = 0.001
inceptionv3_full_Epochs = 2
inceptionv3_full_splits = 4
# Define loss and optimizer
inceptionv3_full_loss_fun = nn.BCEWithLogitsLoss()
inceptionv3_full_optimizer = torch.optim.Adam(inceptionv3_full.parameters(), lr=inceptionv3_full_LR)

In [28]:
result_inceptionv3_fulll_partial_model = train(inceptionv3_full, train_data, train_X, train_y, inceptionv3_full_optimizer, inceptionv3_full_loss_fun, inceptionv3_full_Epochs, inceptionv3_full_splits)

Training
Evaluting
Epoch 1/2 - Train Loss: 0.1383, Train Acc: 0.9418, Val Loss: 0.0936, Val Acc: 0.9690
Evaluting
Epoch 2/2 - Train Loss: 1.2684, Train Acc: 0.5201, Val Loss: 0.6921, Val Acc: 0.5426
Training
Evaluting
Epoch 1/2 - Train Loss: 0.0655, Train Acc: 0.9871, Val Loss: 0.0267, Val Acc: 0.9922
Evaluting
Epoch 2/2 - Train Loss: 0.8681, Train Acc: 0.5899, Val Loss: 0.6929, Val Acc: 0.5426
Training
Evaluting
Epoch 1/2 - Train Loss: 0.1797, Train Acc: 0.9276, Val Loss: 0.0128, Val Acc: 1.0000
Evaluting
Epoch 2/2 - Train Loss: 1.6235, Train Acc: 0.5744, Val Loss: 0.6929, Val Acc: 0.5426
Training
Evaluting
Epoch 1/2 - Train Loss: 0.2183, Train Acc: 0.9109, Val Loss: 0.0303, Val Acc: 0.9883
Evaluting
Epoch 2/2 - Train Loss: 0.9169, Train Acc: 0.5530, Val Loss: 0.6932, Val Acc: 0.4553

Average Validation Accuracy across folds: 0.5208
Average Validation Loss across folds: 11.7771


### 6.1 InceptionV3 partial fine-tuning

In [29]:
# Load the AlexNet model and weights with pre-trian ImageNet weights
from torchvision.models import inception_v3
from torchvision.models import Inception_V3_Weights

# initialize the pretrained alexnet
inceptionv3_partial = inception_v3(weights=Inception_V3_Weights.DEFAULT)

inceptionv3_partial.eval()

Inception3(
  (Conv2d_1a_3x3): BasicConv2d(
    (conv): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2a_3x3): BasicConv2d(
    (conv): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(32, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_2b_3x3): BasicConv2d(
    (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (bn): BatchNorm2d(64, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (maxpool1): MaxPool2d(kernel_size=3, stride=2, padding=0, dilation=1, ceil_mode=False)
  (Conv2d_3b_1x1): BasicConv2d(
    (conv): Conv2d(64, 80, kernel_size=(1, 1), stride=(1, 1), bias=False)
    (bn): BatchNorm2d(80, eps=0.001, momentum=0.1, affine=True, track_running_stats=True)
  )
  (Conv2d_4a_3x3): BasicConv2d(
    (conv): Conv2d(80, 192, kernel_size=(3, 3), stri

### 6.2 Frezze Early layers

In [30]:
inceptionv3_partial.aux_logits = False

for parameter in inceptionv3_partial.parameters():
    parameter.requires_grad = False


weight_params = [
    (name, param)
    for name, param in inceptionv3_partial.named_parameters()
    if name.endswith("weight")
]

for name, param in weight_params[-8:]:
    param.requires_grad = True
    print(f"Unfrozen weight: {name}")

Unfrozen weight: Mixed_7c.branch3x3dbl_2.bn.weight
Unfrozen weight: Mixed_7c.branch3x3dbl_3a.conv.weight
Unfrozen weight: Mixed_7c.branch3x3dbl_3a.bn.weight
Unfrozen weight: Mixed_7c.branch3x3dbl_3b.conv.weight
Unfrozen weight: Mixed_7c.branch3x3dbl_3b.bn.weight
Unfrozen weight: Mixed_7c.branch_pool.conv.weight
Unfrozen weight: Mixed_7c.branch_pool.bn.weight
Unfrozen weight: fc.weight


### 6.3 Modify the head of inceptionV3 

In [31]:
inceptionv3_partial.fc = nn.Sequential(
    Linear(in_features=2048, out_features=1, bias=True),
)

### 6.4 Get the summary of the model

In [32]:
summary(inceptionv3_partial,
        input_size=(16, 3, 299, 299),
        verbose=0,
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=18,
        row_settings=["var_names"]
)

Layer (type (var_name))                  Input Shape        Output Shape       Param #            Trainable
Inception3 (Inception3)                  [16, 3, 299, 299]  [16, 1]            3,326,696          Partial
├─BasicConv2d (Conv2d_1a_3x3)            [16, 3, 299, 299]  [16, 32, 149, 149] --                 False
│    └─Conv2d (conv)                     [16, 3, 299, 299]  [16, 32, 149, 149] (864)              False
│    └─BatchNorm2d (bn)                  [16, 32, 149, 149] [16, 32, 149, 149] (64)               False
├─BasicConv2d (Conv2d_2a_3x3)            [16, 32, 149, 149] [16, 32, 147, 147] --                 False
│    └─Conv2d (conv)                     [16, 32, 149, 149] [16, 32, 147, 147] (9,216)            False
│    └─BatchNorm2d (bn)                  [16, 32, 147, 147] [16, 32, 147, 147] (64)               False
├─BasicConv2d (Conv2d_2b_3x3)            [16, 32, 147, 147] [16, 64, 147, 147] --                 False
│    └─Conv2d (conv)                     [16, 32, 147, 147

In [33]:
# Specifiy learing rate and number of Epochs
inceptionv3_partial_LR = 0.001
inceptionv3_partial_Epochs = 2
inceptionv3_partial_splits = 4
# Define loss and optimizer
inceptionv3_partial_loss_fun = nn.BCEWithLogitsLoss()
inceptionv3_partial_optimizer = torch.optim.Adam(inceptionv3_partial.parameters(), lr=inceptionv3_partial_LR)

In [34]:
result_inceptionv3_partial_model = train(inceptionv3_partial, train_data, train_X, train_y, inceptionv3_partial_optimizer, inceptionv3_partial_loss_fun, inceptionv3_partial_Epochs, inceptionv3_partial_splits)

Training
Evaluting
Epoch 1/2 - Train Loss: 0.2093, Train Acc: 0.9211, Val Loss: 0.3771, Val Acc: 0.8876
Evaluting
Epoch 2/2 - Train Loss: 0.0653, Train Acc: 0.9806, Val Loss: 0.1628, Val Acc: 0.9612
Training
Evaluting
Epoch 1/2 - Train Loss: 0.0947, Train Acc: 0.9651, Val Loss: 0.0728, Val Acc: 0.9729
Evaluting
Epoch 2/2 - Train Loss: 0.0386, Train Acc: 0.9858, Val Loss: 0.0423, Val Acc: 0.9845
Training
Evaluting
Epoch 1/2 - Train Loss: 0.0947, Train Acc: 0.9651, Val Loss: 0.0252, Val Acc: 0.9922
Evaluting
Epoch 2/2 - Train Loss: 0.0123, Train Acc: 1.0000, Val Loss: 0.0048, Val Acc: 1.0000
Training
Evaluting
Epoch 1/2 - Train Loss: 0.0698, Train Acc: 0.9755, Val Loss: 0.0250, Val Acc: 1.0000
Evaluting
Epoch 2/2 - Train Loss: 0.0121, Train Acc: 0.9987, Val Loss: 0.0126, Val Acc: 1.0000

Average Validation Accuracy across folds: 0.9864
Average Validation Loss across folds: 0.9456


In [35]:
from torchmetrics.classification import BinaryPrecision, BinaryRecall, BinaryF1Score

def test(model, test_loader, loss_fn):
    """
    Test the model and return testing metrics
    """
    model.eval()
    print("Testing")
    
    val_loss, correct, total = 0.0, 0, 0
    y_pred, y_true = [], []
    
    # Initialize metrics
    precision = BinaryPrecision()
    recall = BinaryRecall()
    f1 = BinaryF1Score()
    
    with torch.no_grad():
        for images, labels in test_loader:
            predictions = model(images)
            labels = labels.unsqueeze(1).float()
            
   
            val_loss += loss_fn(predictions, labels).item()
            

            preds = (torch.sigmoid(predictions) > 0.5).float()
            
            y_pred.append(preds)
            y_true.append(labels)
            
            
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    
    y_pred = torch.cat(y_pred, dim=0)
    y_true = torch.cat(y_true, dim=0)
    
    # Metrics
    test_accuracy = correct / total
    test_loss = val_loss / len(test_loader)
    test_precision = precision(y_pred, y_true).item()
    test_recall = recall(y_pred, y_true).item()
    test_f1 = f1(y_pred, y_true).item()
    
    return {
        'test_loss': test_loss,
        'test_accuracy': test_accuracy,
        'test_precision': test_precision,
        'test_recall': test_recall,
        'test_f1': test_f1,
    }

In [38]:
inceptionv3_test_results = test(inceptionv3, test_loader, inceptionv3_loss_fun)

Testing


In [39]:
inceptionv3_full_test_results = test(inceptionv3_full, test_loader, inceptionv3_full_loss_fun)

Testing


In [41]:
inceptionv3_partial_test_results = test(inceptionv3_partial, test_loader, inceptionv3_partial_loss_fun)

Testing


In [42]:
inceptionv3_test_results

{'test_loss': 0.22043756975067985,
 'test_accuracy': 0.9069767441860465,
 'test_precision': 1.0,
 'test_recall': 0.7966101765632629,
 'test_f1': 0.8867924809455872}

In [43]:
inceptionv3_full_test_results

{'test_loss': 0.6929188900523715,
 'test_accuracy': 0.4573643410852713,
 'test_precision': 0.45736435055732727,
 'test_recall': 1.0,
 'test_f1': 0.6276595592498779}

In [45]:
inceptionv3_partial_test_results

{'test_loss': 0.014385177129750244,
 'test_accuracy': 0.9922480620155039,
 'test_precision': 0.9833333492279053,
 'test_recall': 1.0,
 'test_f1': 0.9915966391563416}

In [48]:
avg_val_acc = sum(r['val_acc'][-1] for r in result) / inceptionv3_n_splits
avg_val_loss = sum(r['val_loss'][-1] for r in result) / inceptionv3_n_splits
avg_train_acc = sum(r['train_acc'][-1] for r in result) / inceptionv3_n_splits
avg_train_loss = sum(r['train_loss'][-1] for r in result) / inceptionv3_n_splits
print(f"inceptionv3 average Validation Accuracy across folds: {avg_val_acc:.4f}")
print(f"inceptionv3 average Validation Loss across folds: {avg_val_loss:.4f}")
print(f"\inceptionv3 average Training Accuracy across folds: {avg_train_acc:.4f}")
print(f"inceptionv3 average Traing Loss across folds: {avg_train_loss:.4f}")

inceptionv3 average Validation Accuracy across folds: 0.9185
inceptionv3 average Validation Loss across folds: 4.7052
\inceptionv3 average Training Accuracy across folds: 0.9331
inceptionv3 average Traing Loss across folds: 12.4842


In [49]:
avg_val_acc = sum(r['val_acc'][-1] for r in result_inceptionv3_fulll_partial_model) / inceptionv3_n_splits
avg_val_loss = sum(r['val_loss'][-1] for r in result_inceptionv3_fulll_partial_model) / inceptionv3_n_splits
avg_train_acc = sum(r['train_acc'][-1] for r in result_inceptionv3_fulll_partial_model) / inceptionv3_n_splits
avg_train_loss = sum(r['train_loss'][-1] for r in result_inceptionv3_fulll_partial_model) / inceptionv3_n_splits
print(f"inceptionv3 full fine-tuning average Validation Accuracy across folds: {avg_val_acc:.4f}")
print(f"inceptionv3 full fine-tuning  average Validation Loss across folds: {avg_val_loss:.4f}")
print(f"\inceptionv3 full fine-tuning  average Training Accuracy across folds: {avg_train_acc:.4f}")
print(f"inceptionv3 full fine-tuning  average Traing Loss across folds: {avg_train_loss:.4f}")

inceptionv3 full fine-tuning average Validation Accuracy across folds: 0.5208
inceptionv3 full fine-tuning  average Validation Loss across folds: 11.7771
\inceptionv3 full fine-tuning  average Training Accuracy across folds: 0.5593
inceptionv3 full fine-tuning  average Traing Loss across folds: 57.2921


In [50]:
avg_val_acc = sum(r['val_acc'][-1] for r in result_inceptionv3_partial_model) / inceptionv3_n_splits
avg_val_loss = sum(r['val_loss'][-1] for r in result_inceptionv3_partial_model) / inceptionv3_n_splits
avg_train_acc = sum(r['train_acc'][-1] for r in result_inceptionv3_partial_model) / inceptionv3_n_splits
avg_train_loss = sum(r['train_loss'][-1] for r in result_inceptionv3_partial_model) / inceptionv3_n_splits
print(f"inceptionv3 partial fine-tuning average Validation Accuracy across folds: {avg_val_acc:.4f}")
print(f"inceptionv3 partial fine-tuning  average Validation Loss across folds: {avg_val_loss:.4f}")
print(f"\inceptionv3 partial fine-tuning  average Training Accuracy across folds: {avg_train_acc:.4f}")
print(f"inceptionv3 partial fine-tuning  average Traing Loss across folds: {avg_train_loss:.4f}")

inceptionv3 partial fine-tuning average Validation Accuracy across folds: 0.9864
inceptionv3 partial fine-tuning  average Validation Loss across folds: 0.9456
\inceptionv3 partial fine-tuning  average Training Accuracy across folds: 0.9913
inceptionv3 partial fine-tuning  average Traing Loss across folds: 1.5714
